# RAG Evaluation mit DeepEval

Dieses Notebook evaluiert die RAG-Pipeline mit **DeepEval** und **GPT-4o-mini** als Judge-Modell.

### Metriken
| Metrik | Was wird gemessen | Referenzantwort nötig? |
|---|---|---|
| **Faithfulness** | Erfindet das LLM Infos die nicht im Kontext stehen? | Nein |
| **Answer Relevancy** | Beantwortet die Antwort wirklich die Frage? | Nein |
| **Contextual Precision** | Sind die retrieved Chunks tatsächlich relevant? | Nein |
| **Contextual Recall** | Wurden alle relevanten Stellen gefunden? | Ja |

### Kategorien im Fragenkatalog
- **A** – Faktische Detail- und Datenextraktion
- **B** – Methoden- und Begründungsverständnis  
- **C** – Cross-Document-Synthese
- **D** – Struktur- und Orientierungsfragen
- **E** – Anti-Halluzination und Unsicherheitsverhalten
- **F** – Domänenwissen-Transfer

In [1]:
pip install deepeval openai python-dotenv


  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/844.6 kB ? eta -:--:--
   ---------------------------------------- 844.6/844.6 kB 7.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 9.6 MB/s  0:00:00
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   -------------- ------------------------- 3.4/9.5 MB 16.7 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.5 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------  9.4/9.5 MB 17.3 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 14.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 11.0 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ----------------------------------------  0/16 [pywin32]
   -------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.1 requires posthog<6.0.0,>=2.4.0, but you have posthog 7.12.0 which is incompatible.


In [ ]:
# Setup: Projekt-Root setzen
import os
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print('ROOT:', ROOT)

In [ ]:
# API-Key laden
from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import openai
if not os.getenv('OPENAI_API_KEY') or os.getenv('OPENAI_API_KEY') == 'your_api_key_here':
    raise ValueError('OPENAI_API_KEY nicht gesetzt! Bitte in .env eintragen.')

print('API-Key geladen.')

In [ ]:
# Konfiguration

# RAG-Einstellungen
RAG_MODE       = 'hybrid'
RAG_K          = 20
CHUNK_SIZE     = 1200
CHUNK_OVERLAP  = 200
USE_RERANKER   = True
RERANK_TOP_N   = 5

# Judge-Modell (GPT-4o-mini ist günstiger, für Bachelorarbeit ausreichend)
JUDGE_MODEL = 'gpt-4o-mini'   # Alternativ: 'gpt-4o'

# Welche Fragen evaluieren? None = alle 37
# Beispiel für schnellen Test: QUESTION_IDS = ['q1', 'q2', 'q3']
QUESTION_IDS = None

# Schwellwert ab dem eine Metrik als bestanden gilt (0.0 – 1.0)
THRESHOLD = 0.7

print(f'Judge: {JUDGE_MODEL} | RAG: {RAG_MODE}, k={RAG_K}, reranker={USE_RERANKER}, top_n={RERANK_TOP_N}')
print(f'Threshold: {THRESHOLD} | Fragen: {"alle" if QUESTION_IDS is None else QUESTION_IDS}')

In [ ]:
# Fragen laden
from src.evaluation import load_questions

QUESTIONS_PATH = ROOT / 'data' / 'eval' / 'questions.jsonl'
all_questions = load_questions(QUESTIONS_PATH)

if QUESTION_IDS is not None:
    questions = [q for q in all_questions if q['id'] in QUESTION_IDS]
else:
    questions = all_questions

print(f'{len(questions)} Fragen geladen.')
for q in questions:
    print(f"  [{q['id']}] Kategorie {q['category']}: {q['query'][:70]}...")

In [ ]:
# RAG für alle Fragen ausführen und Ergebnisse sammeln
# Hinweis: Dieser Schritt läuft lokal (Ollama + Reranker) und dauert je nach
# Hardware ca. 30–120 Sekunden pro Frage.
import time
from src.evaluation import run_rag_for_eval

rag_results = []

for i, q in enumerate(questions, start=1):
    print(f'[{i}/{len(questions)}] {q["id"]} ({q["category"]}): {q["query"][:60]}...')
    t0 = time.perf_counter()

    result = run_rag_for_eval(
        query=q['query'],
        mode=RAG_MODE,
        k=RAG_K,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        use_reranker=USE_RERANKER,
        rerank_top_n=RERANK_TOP_N,
    )

    dt = time.perf_counter() - t0
    rag_results.append({
        'id': q['id'],
        'category': q['category'],
        'query': q['query'],
        'expected_output': q.get('expected_output', ''),
        'actual_output': result['answer'],
        'retrieval_context': result['retrieval_context'],
        'sources': result['sources'],
        'latency_s': round(dt, 2),
    })
    print(f'  -> {dt:.1f}s | Antwort: {result["answer"][:100]}...')

print(f'\nAlle {len(rag_results)} RAG-Läufe abgeschlossen.')

In [ ]:
# RAG-Ergebnisse zwischenspeichern (ohne DeepEval – nur Antworten + Chunks)
# Kann sofort als Markdown angezeigt werden, auch ohne Evaluation
import json

eval_dir = ROOT / 'data' / 'eval'
eval_dir.mkdir(parents=True, exist_ok=True)

rag_only = []
for r in rag_results:
    chunks_with_source = []
    for i, chunk_text in enumerate(r['retrieval_context']):
        src = r['sources'][i] if i < len(r['sources']) else {}
        chunks_with_source.append({
            'rank': i + 1,
            'text': chunk_text,
            'source': src.get('source') or '–',
            'page': src.get('page'),
        })
    rag_only.append({
        'id': r['id'],
        'category': r['category'],
        'query': r['query'],
        'expected_output': r.get('expected_output', ''),
        'actual_output': r['actual_output'],
        'latency_s': r['latency_s'],
        'top_chunks': chunks_with_source,
        'metrics': [],  # noch keine Metriken
    })

rag_path = eval_dir / 'results_full.json'
with open(rag_path, 'w', encoding='utf-8') as f:
    json.dump(rag_only, f, ensure_ascii=False, indent=2)

print(f'{len(rag_only)} RAG-Ergebnisse gespeichert: {rag_path}')
print('Jetzt kannst du die Markdown-Zelle ausführen um Antworten + Chunks zu sehen.')


In [ ]:
# DeepEval Test Cases erstellen
from deepeval.test_case import LLMTestCase

test_cases = []
for r in rag_results:
    tc = LLMTestCase(
        input=r['query'],
        actual_output=r['actual_output'],
        expected_output=r['expected_output'],
        retrieval_context=r['retrieval_context'],
    )
    test_cases.append(tc)

print(f'{len(test_cases)} Test Cases erstellt.')

In [ ]:
# Metriken konfigurieren mit GPT-4o-mini als Judge
from deepeval.models import GPTModel
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

judge = GPTModel(model=JUDGE_MODEL)

metrics = [
    FaithfulnessMetric(model=judge, threshold=THRESHOLD),
    AnswerRelevancyMetric(model=judge, threshold=THRESHOLD),
    ContextualPrecisionMetric(model=judge, threshold=THRESHOLD),
    ContextualRecallMetric(model=judge, threshold=THRESHOLD),
]

print('Metriken konfiguriert:', [type(m).__name__ for m in metrics])

In [ ]:
# Evaluation ausführen – robust gegen RateLimitError und Timeout
import os
import time
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig

os.environ['DEEPEVAL_DISABLE_TIMEOUTS'] = 'true'
os.environ['DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS'] = '0'

BATCH_SIZE    = 3    # Fragen pro Batch
SLEEP_BETWEEN = 30   # Sekunden Pause zwischen Batches (erhöht für TPM-Limit)
RETRY_WAIT    = 90   # Sekunden warten nach Timeout/RateLimit-Fehler
MAX_RETRIES   = 3

async_cfg = AsyncConfig(run_async=True, throttle_value=10, max_concurrent=1)

all_test_results = []

for i in range(0, len(test_cases), BATCH_SIZE):
    batch = test_cases[i:i + BATCH_SIZE]
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(test_cases) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f'Batch {batch_num}/{total_batches} ({len(batch)} Fragen)...')

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = evaluate(batch, metrics=metrics, async_config=async_cfg)
            all_test_results.extend(result.test_results)
            break  # Erfolg
        except Exception as e:
            err_type = type(e).__name__
            if attempt < MAX_RETRIES:
                print(f'  [{err_type}] Versuch {attempt}/{MAX_RETRIES} fehlgeschlagen. Warte {RETRY_WAIT}s...')
                time.sleep(RETRY_WAIT)
            else:
                print(f'  Batch {batch_num} endgültig fehlgeschlagen: {err_type}')
                raise

    if i + BATCH_SIZE < len(test_cases):
        print(f'  Pause {SLEEP_BETWEEN}s...')
        time.sleep(SLEEP_BETWEEN)

class _FakeEvalResult:
    def __init__(self, test_results):
        self.test_results = test_results

eval_results = _FakeEvalResult(all_test_results)
print(f'\nEvaluation abgeschlossen. {len(all_test_results)} Ergebnisse.')


In [ ]:
# Ergebnisse pro Frage anzeigen
import pandas as pd

rows = []
for r, tc in zip(rag_results, eval_results.test_results):
    row = {
        'ID': r['id'],
        'Kategorie': r['category'],
        'Frage (kurz)': r['query'][:55] + '...',
        'Latenz (s)': r['latency_s'],
    }
    for metric_data in tc.metrics_data:
        name = metric_data.name  # z.B. "Faithfulness", "Answer Relevancy" etc.
        row[name] = round(metric_data.score, 2) if metric_data.score is not None else None
        row[name + ' ✓'] = '✓' if metric_data.success else '✗'
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Begründungen anzeigen — warum hat das Modell diese Note gegeben?
# Optional: FILTER_ID = 'q1'  um nur eine bestimmte Frage zu sehen
FILTER_ID = None   # None = alle Fragen anzeigen

for r, tc in zip(rag_results, eval_results.test_results):
    if FILTER_ID and r['id'] != FILTER_ID:
        continue

    print(f"\n{'='*70}")
    print(f"[{r['id']}] Kategorie {r['category']}")
    print(f"Frage:    {r['query']}")
    print(f"Antwort:  {r['actual_output'][:200]}...")
    print()

    for metric_data in tc.metrics_data:
        status = '✓' if metric_data.success else '✗'
        score  = round(metric_data.score, 2) if metric_data.score is not None else '–'
        print(f"  {status} {metric_data.name}: {score}")
        if metric_data.reason:
            print(f"     Grund: {metric_data.reason}")
        print()

In [ ]:
# Zusammenfassung nach Kategorie
metric_cols = [c for c in df.columns if not c.endswith('✓') and c not in ('ID', 'Kategorie', 'Frage (kurz)', 'Latenz (s)')]
pass_cols   = [c for c in df.columns if c.endswith('✓')]

summary_by_category = df.groupby('Kategorie')[metric_cols].mean().round(2)
summary_total = df[metric_cols].mean().round(2).to_frame(name='Gesamt').T

pass_rates = {}
for col in pass_cols:
    passed = (df[col] == '✓').sum()
    pass_rates[col.replace(' ✓', '')] = f"{passed}/{len(df)} ({100*passed//len(df)}%)"
summary_pass = pd.DataFrame([pass_rates], index=['Bestandsrate'])

print(f'=== Durchschnitt nach Kategorie (Threshold={THRESHOLD}) ===')
print(summary_by_category.to_string())
print(f'\n=== Gesamt-Durchschnitt ===')
print(summary_total.to_string())
print(f'\n=== Bestandsrate ===')
print(summary_pass.to_string())

# Speichern
eval_dir = ROOT / 'data' / 'eval'
summary_by_category.to_csv(eval_dir / 'summary_by_category.csv', encoding='utf-8-sig')
summary_total.to_csv(eval_dir / 'summary_total.csv', encoding='utf-8-sig')
summary_pass.to_csv(eval_dir / 'summary_passrate.csv', encoding='utf-8-sig')
print('\nGespeichert: summary_by_category.csv, summary_total.csv, summary_passrate.csv')

In [ ]:
# Ergebnisse speichern
import json

eval_dir = ROOT / 'data' / 'eval'
eval_dir.mkdir(parents=True, exist_ok=True)

# Tabelle als CSV (zum Öffnen in Excel)
csv_path = eval_dir / 'results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('CSV gespeichert:', csv_path)

# Vollständige Ergebnisse inkl. Begründungen + Top-Chunks als JSON
full_results = []
for r, tc in zip(rag_results, eval_results.test_results):
    # sources und retrieval_context haben denselben Index (beide kommen aus docs)
    chunks_with_source = []
    for i, chunk_text in enumerate(r['retrieval_context']):
        src = r['sources'][i] if i < len(r['sources']) else {}
        source_file = src.get('source') or '–'
        page = src.get('page')
        chunks_with_source.append({
            'rank': i + 1,
            'text': chunk_text,
            'source': source_file,
            'page': page,
        })

    entry = {
        'id': r['id'],
        'category': r['category'],
        'query': r['query'],
        'expected_output': r['expected_output'],
        'actual_output': r['actual_output'],
        'latency_s': r['latency_s'],
        'top_chunks': chunks_with_source,
        'metrics': [
            {
                'name': m.name,
                'score': round(m.score, 4) if m.score is not None else None,
                'success': m.success,
                'reason': m.reason,
            }
            for m in tc.metrics_data
        ],
    }
    full_results.append(entry)

json_path = eval_dir / 'results_full.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)
print('JSON gespeichert:', json_path)


In [ ]:
# Markdown-Report aus results_full.json erzeugen
# Diese Zelle kann unabhängig von der Evaluation ausgeführt werden
import json
import os
import sys
from pathlib import Path

# ROOT bestimmen (falls Zelle isoliert ausgeführt wird)
cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd

eval_dir = ROOT / 'data' / 'eval'
json_path = eval_dir / 'results_full.json'

if not json_path.exists():
    raise FileNotFoundError(f'Keine Ergebnisse gefunden: {json_path}\nBitte zuerst die Evaluation ausführen.')

with open(json_path, 'r', encoding='utf-8') as f:
    full_results = json.load(f)

# Konfigurationsvariablen aus laufender Session oder Fallback
_judge   = globals().get('JUDGE_MODEL', '–')
_mode    = globals().get('RAG_MODE', '–')
_k       = globals().get('RAG_K', '–')
_rerank  = globals().get('USE_RERANKER', '–')
_top_n   = globals().get('RERANK_TOP_N', '–')
_thresh  = globals().get('THRESHOLD', '–')

lines = ['# RAG Evaluation Report\n\n']
lines.append(f'**Anzahl Fragen:** {len(full_results)}  \n')
lines.append(f'**Judge-Modell:** {_judge}  \n')
lines.append(f'**RAG-Modus:** {_mode} | k={_k} | Reranker={_rerank} | top_n={_top_n}  \n')
lines.append(f'**Threshold:** {_thresh}\n\n')
lines.append('---\n\n')

for entry in full_results:
    lines.append(f"## [{entry['id']}] Kategorie {entry['category']}\n\n")
    lines.append(f"**Frage:** {entry['query']}\n\n")
    lines.append(f"**Generierte Antwort:**\n> {entry['actual_output'].strip()}\n\n")

    expected = entry.get('expected_output', '').strip()
    if expected:
        lines.append(f"**Erwartete Antwort:**\n> {expected}\n\n")

    # Metriken-Tabelle
    metrics_list = entry.get('metrics', [])
    if metrics_list:
        lines.append('**Metriken:**\n\n')
        lines.append('| Metrik | Score | Bestanden | Begründung |\n')
        lines.append('|--------|-------|-----------|------------|\n')
        for m in metrics_list:
            score  = f"{m['score']:.2f}" if m['score'] is not None else '–'
            status = '✓' if m['success'] else '✗'
            reason = (m.get('reason') or '').replace('\n', ' ').replace('|', '\\|')
            lines.append(f"| {m['name']} | {score} | {status} | {reason} |\n")
        lines.append('\n')

    # Top Chunks mit Quelldokument
    top_chunks = entry.get('top_chunks', [])[:5]
    if top_chunks:
        lines.append('**Top Chunks (Retrieval Context):**\n\n')
        for chunk in top_chunks:
            src       = chunk.get('source') or '–'
            page      = chunk.get('page')
            src_label = Path(src).name if src != '–' else '–'
            page_str  = f', Seite {page}' if page is not None else ''
            summary   = f"Chunk {chunk['rank']} — {src_label}{page_str}"
            lines.append(f"<details><summary>{summary}</summary>\n\n")
            lines.append(f"```\n{chunk['text'].strip()}\n```\n\n")
            lines.append('</details>\n\n')

    lines.append('---\n\n')

md_path = eval_dir / 'results_report.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.writelines(lines)

print(f'Markdown-Report gespeichert: {md_path}')
print(f'Öffnen: Ctrl+Shift+V in VS Code für die Preview')
